# Model 1 — Leaky Integrate-and-Fire Foundations

This notebook is the guided entry point for the project. The implementation lives in `src/neuron_lab`, while this notebook is for experiments, plots, derivations, and interpretation.

## Learning goals

1. Understand passive membrane charging and decay.
2. Relate resistance and capacitance to the membrane time constant.
3. Distinguish the continuous subthreshold model from the artificial spike event.
4. Observe how input current changes spike timing and firing rate.
5. Check simulation results against analytical expectations.


## Equation and units

$$C_m \frac{dV}{dt} = -\frac{V-E_L}{R_m} + I(t)$$

or

$$\frac{dV}{dt}=\frac{E_L-V+R_m I(t)}{\tau_m}, \qquad \tau_m=R_mC_m.$$

This project uses `ms`, `mV`, `nA`, `MΩ`, and `nF`. Therefore, `MΩ × nA = mV` and `MΩ × nF = ms`.

The threshold/reset mechanism is a modeling convention. It does not reproduce the biological shape of an action potential.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from neuron_lab import LIFParameters, SimulationConfig, simulate_lif, step_current
from neuron_lab.plotting import plot_fi_curve, plot_lif_result


## Inspect the defaults

Before simulating, predict the membrane time constant and passive rheobase from the parameter values.


In [ ]:
parameters = LIFParameters()
parameters


In [ ]:
print(f"Membrane time constant: {parameters.membrane_time_constant_ms:.1f} ms")
print(f"Passive rheobase estimate: {parameters.rheobase_na:.3f} nA")


## Experiment 1: passive, subthreshold response

Apply `0.10 nA`, which is below the passive threshold estimate. Predict the steady-state voltage using:

$$V_\infty = E_L + R_m I.$$


In [ ]:
config = SimulationConfig(duration_ms=250.0, dt_ms=0.1)
subthreshold_current = step_current(
    config,
    amplitude_na=0.10,
    start_ms=25.0,
    stop_ms=175.0,
)
subthreshold_result = simulate_lif(parameters, config, subthreshold_current)
figure, _ = plot_lif_result(subthreshold_result, title="Subthreshold response")
plt.show()


In [ ]:
expected_steady_state_mv = (
    parameters.resting_potential_mv
    + parameters.resistance_mohm * 0.10
)
print(f"Expected passive steady state: {expected_steady_state_mv:.1f} mV")
print(f"Maximum simulated voltage: {subthreshold_result.voltage_mv.max():.2f} mV")


**Write before continuing:** Why does the voltage not change instantaneously when the current begins? Why does it decay after the current ends?


## Experiment 2: threshold, reset, and refractory behavior

Now apply a current above the passive rheobase. Notice that plotted spike peaks are threshold markers rather than action-potential waveforms.


In [ ]:
suprathreshold_current = step_current(
    config,
    amplitude_na=0.22,
    start_ms=25.0,
    stop_ms=225.0,
)
suprathreshold_result = simulate_lif(parameters, config, suprathreshold_current)
figure, _ = plot_lif_result(suprathreshold_result, title="Suprathreshold response")
plt.show()

print(f"Spike count: {suprathreshold_result.spike_count}")
print(f"Spike times (ms): {suprathreshold_result.spike_times_ms}")


## Experiment 3: firing-rate–current curve

Use a family of constant currents to see how a continuous input is converted into a spike rate.


In [ ]:
fi_config = SimulationConfig(duration_ms=1_000.0, dt_ms=0.1)
currents_na = np.linspace(0.0, 0.5, 26)
firing_rates_hz = []

for amplitude_na in currents_na:
    current = step_current(
        fi_config,
        amplitude_na=float(amplitude_na),
        start_ms=100.0,
        stop_ms=900.0,
    )
    result = simulate_lif(parameters, fi_config, current)
    active_duration_seconds = 0.8
    firing_rates_hz.append(result.spike_count / active_duration_seconds)

figure, _ = plot_fi_curve(currents_na, firing_rates_hz)
plt.axvline(parameters.rheobase_na, linestyle="--", label="Passive rheobase")
plt.legend()
plt.show()


## Experiment 4: numerical time-step check

Forward Euler is approximate. Compare first-spike times as `dt` becomes smaller. Qualitative conclusions should be stable even though exact event times shift slightly.


In [ ]:
for dt_ms in (0.5, 0.2, 0.1, 0.05, 0.02):
    check_config = SimulationConfig(duration_ms=200.0, dt_ms=dt_ms)
    current = step_current(
        check_config,
        amplitude_na=0.22,
        start_ms=20.0,
        stop_ms=180.0,
    )
    result = simulate_lif(parameters, check_config, current)
    first_spike = result.spike_times_ms[0] if result.spike_count else np.nan
    print(f"dt={dt_ms:>4.2f} ms -> first spike={first_spike:>6.2f} ms, spikes={result.spike_count}")


## Interpretation prompts

Write brief answers in your own words:

1. What does the membrane time constant mean operationally?
2. Why is rheobase only an asymptotic passive estimate here?
3. Which aspects of a biological neuron does LIF preserve?
4. Which aspects does it discard?
5. When might this abstraction be preferable to a more realistic model?
6. How did changing `dt` affect the output, and why?

Once these answers are clear, add pulse and sinusoidal-input experiments rather than expanding the software architecture.
